In [1]:
# Import required libraries for data manipulation,visualization and machine learning
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [2]:
# Load training, public test and private test datasets
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

# Check the number of rows and columns in each dataset
print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

# Display the first five records to understand the structure of the dataset
train.head()


Train Shape: (10000, 14)
Public Test Shape: (3000, 14)
Private Test Shape: (3000, 13)


,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


In [3]:
# Examine data types and identify columns with missing values
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  int64  
 1   Age                 8520 non-null   float64
 2   Income              9016 non-null   float64
 3   City_Tier           10000 non-null  int64  
 4   Device_Type         10000 non-null  str    
 5   Traffic_Source      10000 non-null  str    
 6   Pages_Viewed        10000 non-null  int64  
 7   Products_Viewed     10000 non-null  int64  
 8   Time_On_Site        8152 non-null   float64
 9   Previous_Purchases  10000 non-null  int64  
 10  Discount_Seen       10000 non-null  int64  
 11  Browser_Version     10000 non-null  int64  
 12  Campaign_Code       10000 non-null  int64  
 13  Converted           10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
memory usage: 1.2 MB


In [4]:
# Missing values summary (Calculate missing values present in each column)
train.isnull().sum()

User_ID                  0
Age                   1480
Income                 984
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          1848
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64

In [5]:
# Analyze class distribution of the target variable
train["Converted"].value_counts()

Converted
0    6913
1    3087
Name: count, dtype: int64

In [6]:
# Compute class proportions
train["Converted"].value_counts(normalize=True)

Converted
0    0.6913
1    0.3087
Name: proportion, dtype: float64

## Statistical Summary of Numerical Features

This section provides descriptive statistics for all numerical variables,
including count, mean, standard deviation, minimum, maximum and quartiles.

The objective is to identify:
- Missing values
- Outliers
- Feature distributions
- Potential relationships with conversion

In [7]:
# Generate descriptive statistics for numerical features
train.describe()

,User_ID,Age,Income,City_Tier,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
count,10000.00000,8520.000000,9016.000000,10000.000000,10000.00000,10000.000000,8152.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,41.457746,69961.772797,1.933400,15.60850,15.219300,13.667241,2.972500,0.540700,12.473900,5511.162200,0.30870
std,2886.89568,13.770164,24790.673822,0.737712,8.62346,8.927563,19.438244,1.738691,0.498366,6.892856,2596.933802,0.46198
min,1.00000,18.000000,12000.000000,1.000000,1.00000,1.000000,0.800000,0.000000,0.000000,1.000000,1000.000000,0.00000
25%,2500.75000,29.000000,52294.644359,1.000000,8.00000,8.000000,7.640000,2.000000,0.000000,7.000000,3251.750000,0.00000
50%,5000.50000,41.000000,70171.613672,2.000000,16.00000,15.000000,11.145000,3.000000,1.000000,12.000000,5515.500000,0.00000
75%,7500.25000,53.000000,86907.747154,2.000000,23.00000,23.000000,15.630000,4.000000,1.000000,18.000000,7759.000000,1.00000
max,10000.00000,65.000000,161687.774167,3.000000,30.00000,37.000000,607.390000,12.000000,1.000000,24.000000,9998.000000,1.00000


##### Observation:
Time_On_Site contains extreme values.
The maximum value (607 minutes) is significantly larger than the median (~11 minutes),
suggesting the presence of outliers.

##### Objective:
Identify behavioural and demographic differences
between converted and non-converted users.

Features showing large differences may be strong
predictors of conversion.

In [8]:
# Compare average feature values between converted and non-converted users
train.groupby("Converted").mean(numeric_only=True)

,User_ID,Age,Income,City_Tier,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code
Converted,,,,,,,,,,,
0,4998.182844,41.548475,69134.659095,1.935918,13.802546,13.360480,12.597556,2.859974,0.505135,12.480255,5525.615507
1,5005.689018,41.253435,71771.595850,1.927762,19.652737,19.381924,15.925828,3.224490,0.620343,12.459670,5478.795594


## Key Findings from Conversion Analysis

Comparing converted and non-converted users reveals several patterns:

1. Converted users view substantially more pages and products.
2. Converted users spend more time on the website.
3. Converted users are more likely to have seen a discount.
4. Previous purchase history appears positively associated with conversion.
5. Age, City Tier and Browser Version show little difference between the two groups.

These observations suggest that engagement-related features may be strong predictors of conversion.

In [9]:
# Compare conversion rates by discount exposure
train.groupby("Discount_Seen")["Converted"].mean()

Discount_Seen
0    0.255171
1    0.354171
Name: Converted, dtype: float64

This analysis helps determine whether exposure to discounts
is associated with a higher probability of conversion.

##### Observation:
Users exposed to discounts exhibit a substantially higher conversion rate
(35.4%) compared to users who did not see discounts (25.5%).

This suggests that discount visibility is an important driver of user conversion.

In [10]:
# Compare conversion rates across different traffic acquisition sources
train.groupby("Traffic_Source")["Converted"].mean().sort_values(ascending=False)

Traffic_Source
Referral        0.381181
Email           0.349359
Social Media    0.303061
Organic         0.291762
Paid Ads        0.277123
Name: Converted, dtype: float64

##### Observation:
Referral traffic demonstrates the highest conversion rate,
followed by Email campaigns.

Paid advertisements produce the lowest conversion rate,
indicating differences in user intent across acquisition channels.

In [11]:
# Compare average engagement metrics by traffic source
train.groupby("Traffic_Source")[[
    "Pages_Viewed",
    "Products_Viewed",
    "Time_On_Site"
]].mean()

,Pages_Viewed,Products_Viewed,Time_On_Site
Traffic_Source,,,
Email,16.148504,15.622863,13.395596
Organic,15.658743,15.228264,14.367940
Paid Ads,15.465802,15.171384,13.470107
Referral,15.524725,15.189560,13.056276
Social Media,15.520823,15.098846,13.398771


##### Observation:

Traffic sources exhibit similar engagement metrics
(Pages Viewed, Products Viewed and Time On Site).

However, conversion rates differ substantially across sources.

This suggests that traffic source itself may be an important predictive feature,
independent of user engagement behaviour.

In [12]:
# Calculate missing values for each feature
train.isnull().sum()

User_ID                  0
Age                   1480
Income                 984
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          1848
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64

## Missing Value Analysis

Before training a machine learning model, missing values must be handled.

The dataset contains missing values in:
- Age
- Income
- Time_On_Site

These missing values will be imputed using the median value of each feature.
Median imputation is preferred because it is robust to outliers.

## Baseline Model Preparation

In [13]:
# Define target variable
TARGET = "Converted"

# Separate features and target
X = train.drop(columns=[TARGET])
y = train[TARGET]

In [14]:
# Remove identifier column
X = X.drop(columns=["User_ID"])
X_private = private_test.drop(columns=["User_ID"])

In [15]:
# Identify numerical and categorical columns
num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

Numerical Columns: ['Age', 'Income', 'City_Tier', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases', 'Discount_Seen', 'Browser_Version', 'Campaign_Code']
Categorical Columns: ['Device_Type', 'Traffic_Source']


## Feature Classification

Features were divided into numerical and categorical groups.

Numerical features will undergo missing value imputation and scaling.

Categorical features will be encoded into numerical representations before model training.

In [16]:
# Split training data into training and validation sets
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

(8000, 12)
(2000, 12)


In [17]:
# Preprocessing utilities
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [18]:
# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [19]:
# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [20]:
# Apply preprocessing to appropriate columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

## Built Logistic Regression Baseline

In [21]:
# Create baseline machine learning pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['Age','Income','City_Tier',...,'Discount_Seen','Browser_Version', 'Campaign_Code']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pas

In [22]:
# Evaluate baseline model
from sklearn.metrics import f1_score

preds = model.predict(X_valid)

f1 = f1_score(y_valid, preds)

print("Validation F1 Score:", f1)

Validation F1 Score: 0.3813559322033898


## Built Random Forest model

In [23]:
## MODEL IMPROVEMENT
# Train a Random Forest for feature importance analysis
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_valid)

from sklearn.metrics import f1_score

print("Random Forest F1:",
      f1_score(y_valid, preds_rf))

Random Forest F1: 0.38358974358974357


## Handled class imbalance

In [24]:
# Balanced Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier",
         RandomForestClassifier(
             n_estimators=300,
             random_state=42,
             class_weight="balanced"
         ))
    ]
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_valid)

from sklearn.metrics import f1_score

print("Balanced RF F1:",
      f1_score(y_valid, preds_rf))

Balanced RF F1: 0.5110941086457537


##### Model Comparison

Three baseline approaches were evaluated:

1. Logistic Regression
2. Random Forest
3. Balanced Random Forest

The Balanced Random Forest achieved the highest validation F1 Score.

This suggests that class imbalance was affecting model performance and that giving greater importance to the minority class improves prediction quality.

In [25]:
from sklearn.metrics import classification_report

print(classification_report(y_valid, preds_rf))

              precision    recall  f1-score   support

           0       0.78      0.74      0.76      1383
           1       0.48      0.54      0.51       617

    accuracy                           0.68      2000
   macro avg       0.63      0.64      0.64      2000
weighted avg       0.69      0.68      0.69      2000



The balanced Random Forest improved minority-class detection.

Although overall accuracy is moderate, the model achieves a substantially higher F1 score by improving recall for converted users.

Since the competition is evaluated using F1 Score, correctly identifying converted users is more important than maximizing overall accuracy.

In [26]:
train.groupby("Device_Type")["Converted"].mean()

Device_Type
Desktop    0.330906
Mobile     0.300629
Tablet     0.306468
Name: Converted, dtype: float64

In [27]:
train.groupby("City_Tier")["Converted"].mean()

City_Tier
1    0.302991
2    0.320337
3    0.294191
Name: Converted, dtype: float64

## Tuned random forest

In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier",
         RandomForestClassifier(
             n_estimators=500,
             max_depth=10,
             min_samples_split=5,
             class_weight="balanced",
             random_state=42
         ))
    ]
)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
preds_rf = rf_model.predict(X_valid)

# Evaluate
f1 = f1_score(y_valid, preds_rf)

print("Tuned Random Forest F1:", f1)

Tuned Random Forest F1: 0.5659121171770972


## Best Model So Far

Model: Tuned Balanced Random Forest

Parameters:
- n_estimators = 500
- max_depth = 10
- min_samples_split = 5
- class_weight = "balanced"

Validation F1 Score: 0.5659

##### Key results
Logistic Regression F1 = 0.381

Balanced Random Forest F1 = 0.511

Tuned Balanced Random Forest F1 = 0.5659

Public Test F1 = 0.5266

## Final Model Training

In [29]:
from sklearn.ensemble import RandomForestClassifier

best_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier",
         RandomForestClassifier(
             n_estimators=500,
             max_depth=10,
             min_samples_split=5,
             class_weight="balanced",
             random_state=42
         ))
    ]
)

In [30]:
best_model.fit(X, y)

print("Final model trained successfully!")

Final model trained successfully!


In [31]:
sample_submission = pd.read_csv("sample_submission.csv")

sample_submission.head()

,User_ID,Converted
0,103001,0
1,103002,0
2,103003,0
3,103004,0
4,103005,0


In [32]:
best_model.fit(X, y)

print("Final model trained!")

Final model trained!


In [33]:
# generate predictions for private_test
private_predictions = best_model.predict(X_private)

print(private_predictions[:10])

[0 1 1 1 0 0 1 0 0 0]


In [34]:
# create submission dataframe
submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": private_predictions
})

submission.head()

,User_ID,Converted
0,103001,0
1,103002,1
2,103003,1
3,103004,1
4,103005,0


In [35]:
submission.shape

(3000, 2)

In [36]:
submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

submission.csv created successfully!


In [37]:
submission["Converted"].value_counts()

Converted
0    1711
1    1289
Name: count, dtype: int64

## Public test evaluation

In [38]:
public_test.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,100001,50.0,65922.413596,3,Tablet,Organic,15,16,NaN,2,0,13,5698,1
1,100002,39.0,43205.865640,1,Mobile,Paid Ads,17,16,12.72,2,1,10,7011,0
2,100003,NaN,44333.908253,3,Mobile,Organic,17,15,3.13,5,0,21,8265,1
3,100004,22.0,141906.816221,3,Desktop,Social Media,1,2,25.32,3,0,9,5592,0
4,100005,47.0,39521.673456,1,Mobile,Social Media,2,4,NaN,3,1,9,4128,0


In [39]:
X_public = public_test.drop(columns=["Converted"])
y_public = public_test["Converted"]

In [40]:
best_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['Age','Income','City_Tier',...,'Discount_Seen','Browser_Version', 'Campaign_Code']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pas

In [41]:
public_preds = best_model.predict(X_public)

In [42]:
from sklearn.metrics import f1_score

public_f1 = f1_score(y_public, public_preds)

print("Public Test F1:", public_f1)

Public Test F1: 0.5265567765567766


In [43]:
from sklearn.metrics import classification_report

print(classification_report(y_public, public_preds))

              precision    recall  f1-score   support

           0       0.82      0.66      0.73      2114
           1       0.44      0.65      0.53       886

    accuracy                           0.66      3000
   macro avg       0.63      0.65      0.63      3000
weighted avg       0.71      0.66      0.67      3000



## Generate the final submission

In [44]:
best_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['Age','Income','City_Tier',...,'Discount_Seen','Browser_Version', 'Campaign_Code']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pas

In [45]:
private_predictions = best_model.predict(X_private)

In [46]:
submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": private_predictions
})

submission.to_csv("submission.csv", index=False)

In [47]:
# checking before submission
submission.head()

,User_ID,Converted
0,103001,0
1,103002,1
2,103003,1
3,103004,1
4,103005,0


In [48]:
submission.columns

Index(['User_ID', 'Converted'], dtype='str')

In [49]:
submission.shape

(3000, 2)

In [50]:
submission["Converted"].value_counts()

Converted
0    1711
1    1289
Name: count, dtype: int64